In [ ]:
import os
import zipfile
import urllib.request

# Alternative URL for the dataset
url_alt = "https://storage.googleapis.com/kaggle-datasets/kaggle/359/dogs-vs-cats/cats_and_dogs_filtered.zip?resource=download"
zip_path = "/content/cats_and_dogs_filtered.zip"

print(f"Attempting to download from: {url_alt}")

# Due to Kaggle's direct download policies, it is generally recommended to use the Kaggle API.
# However, if an alternative direct link is found that works, it can be used.
# The previous Google Storage links seem to be problematic.
# Let's try to get the file using a different approach if the direct link fails again,
# for now, using a potential direct download link from a similar source if available.

# Note: Direct download links from Kaggle can be temporary or require authentication.
# A more robust solution might involve using the Kaggle API or manually uploading the dataset.
# For demonstration, I'm providing a commonly used alternative link for this dataset.

try:
    urllib.request.urlretrieve(url_alt, zip_path)
    print("Download completed. Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content")

    print("Extraction complete.")
    print(os.listdir("/content/cats_and_dogs_filtered"))
except Exception as e:
    print(f"An error occurred during download or extraction: {e}")
    print("It seems direct download links for this dataset are unstable. \nConsider using the Kaggle API (requires authentication) or manually uploading the dataset.")
    print("For example, using Kaggle API: !pip install kaggle && !kaggle datasets download -d chetankv/dogs-vs-cats && !unzip dogs-vs-cats.zip")

In [ ]:
!pip install datasets --quiet

In [ ]:
from datasets import load_dataset

hf_veri = load_dataset("Bingsu/Cat_and_Dog")
print(hf_veri)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Görüntüleri sabit boyuta getirip tensöre çeviriyoruz
donusum = transforms.Compose([
    transforms.Resize((64, 64)),       # Tüm görüntüleri 64x64'e sabitliyoruz
    transforms.ToTensor(),             # PIL görüntüyü [0,1] aralığında tensöre çevirir
    transforms.Normalize(              # RGB kanalları için standart normalizasyon
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class KediKopekDataset(Dataset):
    def __init__(self, hf_split, donusum):
        self.veri = hf_split
        self.donusum = donusum

    def __len__(self):
        return len(self.veri)

    def __getitem__(self, idx):
        ornek = self.veri[idx]
        goruntu = ornek["image"].convert("RGB")  # Bazı görüntüler gri/CMYK olabilir, RGB'ye zorluyoruz
        etiket = ornek["labels"]
        goruntu = self.donusum(goruntu)
        return goruntu, etiket

egitim_dataset = KediKopekDataset(hf_veri["train"], donusum)
test_dataset = KediKopekDataset(hf_veri["test"], donusum)

egitim_loader = DataLoader(egitim_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Eğitim örneği sayısı: {len(egitim_dataset)}")
print(f"Test örneği sayısı  : {len(test_dataset)}")

# Bir örnek batch'in boyutunu kontrol edelim
ornek_goruntuler, ornek_etiketler = next(iter(egitim_loader))
print(f"Batch görüntü boyutu: {ornek_goruntuler.shape}")  # (32, 3, 64, 64) beklenir
print(f"Batch etiket boyutu : {ornek_etiketler.shape}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KediKopekCNN(nn.Module):
    """
    Kedi/Köpek ikili sınıflandırma CNN'i.
    Giriş: (batch, 3, 64, 64) RGB görüntü
    Çıkış: (batch, 1) tek bir logit (0=kedi, 1=köpek olasılığına yakın)
    """
    def __init__(self):
        super().__init__()

        # --- 1. Konvolüsyon Bloğu ---
        # in_channels=3: RGB görüntü (MNIST'te 1'di, burada 3)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        # --- 2. Konvolüsyon Bloğu ---
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        # --- 3. Konvolüsyon Bloğu (gerçek fotoğraflar daha karmaşık, bir blok daha ekliyoruz) ---
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # 3 kez pool uygulanacağı için: 64 -> 32 -> 16 -> 8
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.dropout = nn.Dropout(0.4)          # Az veri olduğu için biraz daha güçlü dropout
        self.fc2 = nn.Linear(128, 1)             # Tek çıktı: ikili sınıflandırma için

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool(x)          # -> (batch, 16, 32, 32)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool(x)          # -> (batch, 32, 16, 16)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool(x)          # -> (batch, 64, 8, 8)

        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)           # -> (batch, 1) ham logit

        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_cnn = KediKopekCNN().to(device)
print(model_cnn)

# Az önceki gerçek batch ile forward pass test edelim
ornek_goruntuler = ornek_goruntuler.to(device)
cikti = model_cnn(ornek_goruntuler)

print(f"Girdi boyutu : {ornek_goruntuler.shape}")
print(f"Çıktı boyutu : {cikti.shape}  # (batch_size=32, 1)")

In [ ]:
# İkili sınıflandırma için: BCEWithLogitsLoss, Sigmoid + Binary Cross Entropy'yi tek adımda,
# sayısal olarak daha kararlı bir şekilde birleştirir
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

EPOCH_SAYISI = 5

for epoch in range(EPOCH_SAYISI):
    model_cnn.train()
    toplam_loss = 0.0

    for gorseller, etiketler in egitim_loader:
        gorseller = gorseller.to(device)
        etiketler = etiketler.to(device).float().unsqueeze(1)  # (batch,) -> (batch, 1), float'a çevir

        optimizer.zero_grad()
        cikti = model_cnn(gorseller)
        loss = criterion(cikti, etiketler)
        loss.backward()
        optimizer.step()

        toplam_loss += loss.item()

    ortalama_loss = toplam_loss / len(egitim_loader)
    print(f"Epoch {epoch+1}/{EPOCH_SAYISI} - Ortalama Loss: {ortalama_loss:.4f}")

In [ ]:
model_cnn.eval()

dogru_sayisi = 0
toplam_sayisi = 0

with torch.no_grad():
    for gorseller, etiketler in test_loader:
        gorseller = gorseller.to(device)
        etiketler = etiketler.to(device).float().unsqueeze(1)

        cikti = model_cnn(gorseller)
        olasiliklar = torch.sigmoid(cikti)          # Logit'i [0,1] olasılığa çevir
        tahminler = (olasiliklar > 0.5).float()      # 0.5 eşiğiyle 0/1 karar ver

        dogru_sayisi += (tahminler == etiketler).sum().item()
        toplam_sayisi += etiketler.size(0)

dogruluk = 100 * dogru_sayisi / toplam_sayisi
print(f"Test doğruluğu: {dogruluk:.2f}%  ({dogru_sayisi}/{toplam_sayisi})")

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

yuklenen = files.upload()  # Bir dosya seçme penceresi açılacak, bilgisayarından bir fotoğraf seç

dosya_adi = list(yuklenen.keys())[0]
goruntu = Image.open(dosya_adi).convert("RGB")

# Görüntüyü göster
plt.imshow(goruntu)
plt.axis("off")
plt.show()

# Modele vermeden önce aynı dönüşümü uyguluyoruz (Resize, ToTensor, Normalize)
girdi_tensor = donusum(goruntu).unsqueeze(0).to(device)  # unsqueeze: (3,64,64) -> (1,3,64,64) batch boyutu ekle

model_cnn.eval()
with torch.no_grad():
    cikti = model_cnn(girdi_tensor)
    olasilik = torch.sigmoid(cikti).item()

if olasilik > 0.5:
    print(f"Tahmin: KÖPEK 🐶  (güven: {olasilik*100:.1f}%)")
else:
    print(f"Tahmin: KEDİ 🐱  (güven: {(1-olasilik)*100:.1f}%)")

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

yuklenen = files.upload()  # Bir dosya seçme penceresi açılacak, bilgisayarından bir fotoğraf seç

dosya_adi = list(yuklenen.keys())[0]
goruntu = Image.open(dosya_adi).convert("RGB")

# Görüntüyü göster
plt.imshow(goruntu)
plt.axis("off")
plt.show()

# Modele vermeden önce aynı dönüşümü uyguluyoruz (Resize, ToTensor, Normalize)
girdi_tensor = donusum(goruntu).unsqueeze(0).to(device)  # unsqueeze: (3,64,64) -> (1,3,64,64) batch boyutu ekle

model_cnn.eval()
with torch.no_grad():
    cikti = model_cnn(girdi_tensor)
    olasilik = torch.sigmoid(cikti).item()

if olasilik > 0.5:
    print(f"Tahmin: KÖPEK 🐶  (güven: {olasilik*100:.1f}%)")
else:
    print(f"Tahmin: KEDİ 🐱  (güven: {(1-olasilik)*100:.1f}%)")